In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import cv2
import random
from torch.amp import autocast, GradScaler

In [ ]:
class UITViICDataset(Dataset):
    def __init__(self, data_file, img_dir, processor, max_length=50):
        self.df = pd.read_csv(data_file)
        self.img_dir = img_dir
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        image_file = self.df.iloc[index]['image_file']
        caption = self.df.iloc[index]['caption']
        
        image_path = os.path.join(self.img_dir, image_file)
        
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        encoding = self.processor(
            images=image,
            text=caption,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        labels = encoding["input_ids"].clone()
        pad_token_id = self.processor.tokenizer.pad_token_id
        labels[labels == pad_token_id] = -100

        encoding["labels"] = labels
        encoding["raw_captions"] = caption 
        
        return encoding

In [ ]:
ROOT_DIR = "./data"
TRAIN_PATH = os.path.join(ROOT_DIR, "cleaned_train.csv")
DEV_PATH = os.path.join(ROOT_DIR, "cleaned_dev.csv")
TEST_PATH = os.path.join(ROOT_DIR, "cleaned_test.csv")
IMAGES_DIR = os.path.join(ROOT_DIR, "images_resized")

In [ ]:
from transformers import BlipProcessor

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")

train_dataset = UITViICDataset(
    data_file=TRAIN_PATH,
    img_dir=IMAGES_DIR,
    processor=processor,
    max_length=50
)

dev_dataset = UITViICDataset(
    data_file=DEV_PATH,
    img_dir=IMAGES_DIR,
    processor=processor,
    max_length=50
)
test_dataset = UITViICDataset(
    data_file=TEST_PATH,
    img_dir=IMAGES_DIR,
    processor=processor,
    max_length=50
)

BATCH_SIZE = 64
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

dev_dataloader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
batch = next(iter(train_dataloader))
print("Các key trong batch:", batch.keys())
print("Kích thước ma trận ảnh:", batch['pixel_values'].shape)
print("Kích thước ma trận text:", batch['input_ids'].shape)

In [ ]:
from transformers import BlipForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device {device}")

model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
model.to(device)

In [ ]:
for param in model.vision_model.parameters():
    param.requires_grad= False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total param: {total_params}")
print(f"Trainable param: {trainable_params}")
print(f"Trainable rate: {trainable_params/total_params:.2f}")


In [ ]:
from torch.optim import AdamW

optimizer_grouped_parameters = [
    p for p in model.parameters() if p.requires_grad
]
optimizer = AdamW(optimizer_grouped_parameters, lr=5e-5)


In [ ]:
import os
import torch
from tqdm import tqdm

EPOCHS = 10  
ACCUMULATION_STEPS = 8
PRINT_EVERY = 50
SAVE_DIR = "./saved_models"

os.makedirs(SAVE_DIR, exist_ok=True)
best_val_loss = float('inf')
scaler = GradScaler()

print("Training start...")

for epoch in range(EPOCHS):
    print(f"\nEpoch: {epoch+1}/{EPOCHS}")
    
    model.train()
    running_loss = 0.0
    train_loss_total = 0.0
    optimizer.zero_grad()

    progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc="Training")
    
    for step, batch in progress_bar:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch["labels"].to(device)

        with autocast("cuda"):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss 
        
        scaler.scale(loss).backward()
        
        running_loss += loss.item()
        train_loss_total += loss.item()

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        if (step + 1) % PRINT_EVERY == 0:
            avg_loss = running_loss / PRINT_EVERY
            progress_bar.set_postfix({'loss': f"{avg_loss:.4f}"})
            running_loss = 0.0
            
    avg_train_loss = train_loss_total / len(train_dataloader)


    model.eval()
    val_loss_total = 0.0
    
    val_progress_bar = tqdm(dev_dataloader, total=len(dev_dataloader), desc="Validating")
    with torch.no_grad():
        for batch in val_progress_bar:
            with autocast("cuda"):
                outputs = model(
                    pixel_values=batch['pixel_values'].to(device),
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    labels=batch["labels"].to(device)
                )
            val_loss_total += outputs.loss.item()
            
    avg_val_loss = val_loss_total / len(dev_dataloader)
    print(f"-> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    if avg_val_loss < best_val_loss:
        print(f"Val loss cải thiện từ {best_val_loss:.4f} xuống {avg_val_loss:.4f}. Đang lưu model...")
        best_val_loss = avg_val_loss
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)

In [ ]:
def evaluate_test_set(model, processor, test_dataloader, device):
    model.eval()
    generated_captions = []
    
    print("\nStarting Test Inference...")
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            pixel_values = batch['pixel_values'].to(device)
            
            generated_ids = model.generate(
                pixel_values=pixel_values,
                max_length=50,
                num_beams=3,
                early_stopping=True
            )
            
            preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
            generated_captions.extend(preds)
            
    return generated_captions

test_predictions = evaluate_test_set(model, processor, test_dataloader, device)

for i in range(5):
    print(f"Ảnh {i+1}: {test_predictions[i]}")

In [ ]:
import torch
import evaluate
from tqdm import tqdm

def get_predictions_and_references(model, processor, test_dataloader, device):
    model.eval()
    predictions = []
    references = []
    
    print("Generating captions for test set...")
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            pixel_values = batch['pixel_values'].to(device)
            
            raw_caps = batch['raw_captions'] 
            
            generated_ids = model.generate(
                pixel_values=pixel_values,
                max_length=50,
                num_beams=3,
                early_stopping=True
            )
            
            preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
            
            predictions.extend(preds)
            references.extend([[cap] for cap in raw_caps]) 
            
    return predictions, references

In [ ]:
def calculate_metrics(predictions, references):
    bleu_metric = evaluate.load("bleu")
    rouge_metric = evaluate.load("rouge")
    meteor_metric = evaluate.load("meteor")
    
    print("Calculating metrics...")
    
    bleu_results = bleu_metric.compute(predictions=predictions, references=references)
    
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    
    meteor_results = meteor_metric.compute(predictions=predictions, references=references)
    
    print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH")
    print(f"BLEU-4 Score : {bleu_results['bleu'] * 100:.2f}")
    print(f"ROUGE-L Score: {rouge_results['rougeL'] * 100:.2f}")
    print(f"METEOR Score : {meteor_results['meteor'] * 100:.2f}")
    
    return {
        "bleu": bleu_results,
        "rouge": rouge_results,
        "meteor": meteor_results
    }

predictions, references = get_predictions_and_references(model, processor, test_dataloader, device)
metrics = calculate_metrics(predictions, references)